# SEPA — Evolución del Costo de la Canasta Representativa

**Objetivo:** Calcular el costo mensual de una **canasta fija de 51 productos** a lo largo del tiempo (enero 2024 – presente), usando los archivos semestrales del SEPA.

**Canasta:** selección manual del economista basada en los candidatos de `exploracion_productos.ipynb`. Familia tipo 4 integrantes (2 adultos + 2 hijos), con cantidades mensuales representativas.

**Estructura del notebook:**
1. Configuración (canasta + rutas)
2. Instalación de dependencias e imports
3. Funciones de carga (semestral → filtrado por EAN)
4. Procesamiento de todos los semestres → serie mensual de precios
5. Verificación de cobertura (qué productos faltan en qué meses)
6. Cálculo del costo mensual de la canasta
7. Visualizaciones: evolución nominal, variación mensual, desglose por categoría
8. Comparación con IPC INDEC
9. Exportación a Excel

**Fuente de datos:** mismos ZIPs semestrales que `exploracion_productos.ipynb` (`2024A.zip`, `2024B.zip`, `2025A.zip`, `2025B.zip`, `2026A.zip`).

> **Nota sobre precios:** los datos SEPA pueden estar en centavos (pre-2025) o en pesos (2025B+). El notebook **autodetecta el factor por semestre** y normaliza a pesos argentinos antes de cualquier cálculo.

In [ ]:
# ===========================================================
# CONFIGURACIÓN — Solo modificar esta sección
# ===========================================================

SEPA_SOURCE = 'mi_drive'   # 'mi_drive' | 'local'

SEPA_DIR   = '/content/drive/MyDrive/carga'
OUTPUT_DIR = '/content/drive/MyDrive/carga/output_canasta'

# Usar caché de parquets para no reprocesar si el kernel de Colab crashea
USE_CACHE = True

# ===========================================================
# CANASTA REPRESENTATIVA — Familia tipo 4 integrantes
# Fuente: candidatos SEPA abril 2026 + criterio económico
# Cantidades: unidades por mes
# ===========================================================
CANASTA = [
    # ── LÁCTEOS ──────────────────────────────────────────────────────────────
    {'ean': '7790742363008', 'descripcion': 'Leche La Serenísima Entera 1L',        'categoria': 'Lácteos',            'cantidad': 20},
    {'ean': '7793940054006', 'descripcion': 'Manteca La Serenísima 200g',            'categoria': 'Lácteos',            'cantidad':  2},
    {'ean': '7790742625304', 'descripcion': 'Dulce de Leche La Serenísima 400g',     'categoria': 'Lácteos',            'cantidad':  2},
    {'ean': '7791337061361', 'descripcion': 'Queso Crema Casancrem 290g',            'categoria': 'Lácteos',            'cantidad':  2},
    {'ean': '7791337007611', 'descripcion': 'Yogur Firme Frutilla Yogurisimo 190g',  'categoria': 'Lácteos',            'cantidad':  8},
    # ── CEREALES Y DERIVADOS ─────────────────────────────────────────────────
    {'ean': '7790070337009', 'descripcion': 'Fideos Tallarines Lucchetti 500g',      'categoria': 'Cereales y derivados', 'cantidad': 4},
    {'ean': '7790070336293', 'descripcion': 'Fideos Tirabuzón Matarazzo 500g',       'categoria': 'Cereales y derivados', 'cantidad': 4},
    {'ean': '7791120031557', 'descripcion': 'Arroz Molinos Ala 1kg',                 'categoria': 'Cereales y derivados', 'cantidad': 2},
    {'ean': '7792180139320', 'descripcion': 'Harina Cañuelas 1kg',                   'categoria': 'Cereales y derivados', 'cantidad': 2},
    {'ean': '7790040143234', 'descripcion': 'Galletitas Chocolinas 262g',            'categoria': 'Cereales y derivados', 'cantidad': 2},
    {'ean': '7622201735258', 'descripcion': 'Galletitas Oreo 354g',                  'categoria': 'Cereales y derivados', 'cantidad': 1},
    {'ean': '7790070621801', 'descripcion': 'Ravioles La Salteña 900g',              'categoria': 'Cereales y derivados', 'cantidad': 2},
    # ── ACEITES ──────────────────────────────────────────────────────────────
    {'ean': '7790070012050', 'descripcion': 'Aceite Girasol Cocinero 900ml',         'categoria': 'Aceites',            'cantidad':  3},
    # ── AZÚCAR, DULCES Y CONSERVAS ───────────────────────────────────────────
    {'ean': '7792540250450', 'descripcion': 'Azúcar Ledesma 1kg',                   'categoria': 'Azúcar y dulces',    'cantidad':  2},
    {'ean': '7793360131516', 'descripcion': 'Mermelada Durazno La Campagnola 390g',  'categoria': 'Azúcar y dulces',    'cantidad':  2},
    {'ean': '7793360131530', 'descripcion': 'Mermelada Frutilla La Campagnola 390g', 'categoria': 'Azúcar y dulces',    'cantidad':  1},
    # ── INFUSIONES ───────────────────────────────────────────────────────────
    {'ean': '7790387013610', 'descripcion': 'Yerba Taragüí 1kg',                    'categoria': 'Infusiones',         'cantidad':  2},
    {'ean': '8445291082199', 'descripcion': 'Café Dolca 100g',                      'categoria': 'Infusiones',         'cantidad':  2},
    {'ean': '7790387800142', 'descripcion': 'Té Taragüí 50 saquitos',               'categoria': 'Infusiones',         'cantidad':  1},
    # ── BEBIDAS ──────────────────────────────────────────────────────────────
    {'ean': '7790895000232', 'descripcion': 'Coca Cola Lata 354cc',                 'categoria': 'Bebidas',            'cantidad':  6},
    {'ean': '7790895000270', 'descripcion': 'Sprite 2.25L',                         'categoria': 'Bebidas',            'cantidad':  4},
    {'ean': '7790895001017', 'descripcion': 'Fanta 2.25L',                          'categoria': 'Bebidas',            'cantidad':  2},
    {'ean': '7790895640476', 'descripcion': 'Aquarius Pera 1.5L',                   'categoria': 'Bebidas',            'cantidad':  4},
    # ── CONDIMENTOS Y ADEREZOS ───────────────────────────────────────────────
    {'ean': '7794000003408', 'descripcion': 'Mayonesa Hellmanns 320g',              'categoria': 'Condimentos',        'cantidad':  2},
    {'ean': '7794000006485', 'descripcion': 'Mostaza Savora 500g',                  'categoria': 'Condimentos',        'cantidad':  1},
    {'ean': '7794000006188', 'descripcion': 'Ketchup Hellmanns 250g',               'categoria': 'Condimentos',        'cantidad':  1},
    {'ean': '7791004000051', 'descripcion': 'Sal Celusal 1kg',                      'categoria': 'Condimentos',        'cantidad':  1},
    {'ean': '7792900093246', 'descripcion': 'Vinagre Dos Anclas 1L',                'categoria': 'Condimentos',        'cantidad':  1},
    {'ean': '7794000008533', 'descripcion': 'Caldo Knorr 12 un',                    'categoria': 'Condimentos',        'cantidad':  1},
    # ── PROTEÍNAS ────────────────────────────────────────────────────────────
    {'ean': '7798092353731', 'descripcion': 'Huevos Carnave Cartón 6 un',           'categoria': 'Proteínas',          'cantidad':  4},
    {'ean': '7790670052791', 'descripcion': 'Hamburguesas Paty 250g',               'categoria': 'Proteínas',          'cantidad':  4},
    {'ean': '7790580131357', 'descripcion': 'Atún La Campagnola 170g',              'categoria': 'Proteínas',          'cantidad':  3},
    {'ean': '7790079018367', 'descripcion': 'Leberwurst Paladini 200g',             'categoria': 'Proteínas',          'cantidad':  2},
    # ── TOMATE Y LEGUMBRES ───────────────────────────────────────────────────
    {'ean': '7790580138868', 'descripcion': 'Puré de Tomate La Campagnola 530g',    'categoria': 'Tomate y legumbres', 'cantidad':  3},
    {'ean': '7790580567101', 'descripcion': 'Tomate en lata Arcor 400g',            'categoria': 'Tomate y legumbres', 'cantidad':  2},
    {'ean': '7793360132483', 'descripcion': 'Porotos Alubia La Campagnola 300g',    'categoria': 'Tomate y legumbres', 'cantidad':  2},
    # ── LIMPIEZA DEL HOGAR ───────────────────────────────────────────────────
    {'ean': '7793253006709', 'descripcion': 'Lavandina Ayudín 2L',                  'categoria': 'Limpieza',           'cantidad':  2},
    {'ean': '7791290794061', 'descripcion': 'Detergente Cif 500ml',                 'categoria': 'Limpieza',           'cantidad':  2},
    {'ean': '7791290792074', 'descripcion': 'Jabón Polvo Ala 3kg',                  'categoria': 'Limpieza',           'cantidad':  1},
    {'ean': '7793253005221', 'descripcion': 'Desinfectante Ayudín 500ml',           'categoria': 'Limpieza',           'cantidad':  1},
    {'ean': '7790990001790', 'descripcion': 'Jabón Zorro 150g',                     'categoria': 'Limpieza',           'cantidad':  4},
    {'ean': '7790117000231', 'descripcion': 'Bolsas Residuos Asurín 30 un',         'categoria': 'Limpieza',           'cantidad':  2},
    # ── HIGIENE PERSONAL ─────────────────────────────────────────────────────
    {'ean': '7791293051208', 'descripcion': 'Jabón Dove 90g',                       'categoria': 'Higiene personal',   'cantidad':  4},
    {'ean': '7509546686516', 'descripcion': 'Crema Dental Colgate 180g',            'categoria': 'Higiene personal',   'cantidad':  2},
    {'ean': '7891024130940', 'descripcion': 'Enjuague Bucal Plax 250ml',            'categoria': 'Higiene personal',   'cantidad':  1},
    {'ean': '7791293050607', 'descripcion': 'Desodorante Axe 230ml',                'categoria': 'Higiene personal',   'cantidad':  1},
    {'ean': '7791293048505', 'descripcion': 'Antitranspirante Dove 150ml',          'categoria': 'Higiene personal',   'cantidad':  1},
    {'ean': '7791070000696', 'descripcion': 'Papel Higiénico Campanita 4x120m',     'categoria': 'Higiene personal',   'cantidad':  3},
    {'ean': '7791290793606', 'descripcion': 'Suavizante Comfort 1L',                'categoria': 'Higiene personal',   'cantidad':  2},
    # ── EXTRA ────────────────────────────────────────────────────────────────
    {'ean': '8445291121904', 'descripcion': 'Cacao Nesquik 800g',                   'categoria': 'Extra',              'cantidad':  1},
    {'ean': '7790580138721', 'descripcion': 'Polenta Prestopronta 730g',            'categoria': 'Extra',              'cantidad':  2},
]

In [ ]:
# Montar Google Drive (solo para SEPA_SOURCE = 'mi_drive')
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive montado en /content/drive')
except ImportError:
    print('Entorno local detectado')

# Instalar dependencias
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', 'tqdm', 'pyarrow', '-q'], check=False)

import zipfile, gzip, re, shutil, warnings, gc
import requests
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Resolver paths ────────────────────────────────────────────────────────────
SEPA_DIR   = Path(SEPA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = OUTPUT_DIR / '_cache'
if USE_CACHE:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

TMP_DIR = Path('/content/tmp_sepa_canasta')
TMP_DIR.mkdir(exist_ok=True)

# ── Lookups por EAN ───────────────────────────────────────────────────────────
df_canasta_def = pd.DataFrame(CANASTA)
df_canasta_def['ean'] = df_canasta_def['ean'].str.zfill(13)
EANS_SET      = set(df_canasta_def['ean'])
EAN_CANTIDAD  = df_canasta_def.set_index('ean')['cantidad'].to_dict()
EAN_DESC      = df_canasta_def.set_index('ean')['descripcion'].to_dict()
EAN_CAT       = df_canasta_def.set_index('ean')['categoria'].to_dict()

print(f'Canasta: {len(EANS_SET)} productos únicos')
print(f'Unidades/mes totales: {df_canasta_def["cantidad"].sum()}')
print(f'\nProductos por categoría:')
print(df_canasta_def.groupby('categoria').agg(
    n_productos=('ean', 'count'),
    unidades=('cantidad', 'sum')
).to_string())
print(f'\nSalida: {OUTPUT_DIR}')
print(f'Caché:  {"habilitado → " + str(CACHE_DIR) if USE_CACHE else "deshabilitado"}')

## 3. Funciones de carga

Estrategia anti-RAM: como la canasta tiene solo 51 productos (~0.03% del universo SEPA), filtramos `id_producto` **antes de acumular datos**. El resultado por semestre es mínimo y cabe cómodamente en RAM.

In [ ]:
_PATRON_SEM = re.compile(r'^(\d{4})(A|B)$', re.IGNORECASE)
_PATRON_ARC = re.compile(r'^(\d{2})(\d{4})_pais_parte.*COMPLETO.*\.csv\.gz$')


def detectar_semestres(sepa_dir: Path) -> list:
    """
    Escanea sepa_dir buscando ZIPs semestrales (ej: 2024A.zip, 2025B.zip).
    Retorna lista de (zip_path, semestre_label) ordenada cronológicamente.
    """
    result = []
    for zip_path in sorted(sepa_dir.glob('*.zip')):
        m = _PATRON_SEM.match(zip_path.stem)
        if m:
            year, half = m.group(1), m.group(2).upper()
            result.append((zip_path, f'{year}-{half}'))
    return result


def _archivos_por_mes(zip_path: Path) -> dict:
    """
    Lee el índice del ZIP y retorna dict: (anio, mes) -> [archivos del mes].
    """
    meses = {}
    with zipfile.ZipFile(zip_path) as z:
        for nombre in z.namelist():
            base = Path(nombre).name
            m = _PATRON_ARC.match(base)
            if not m:
                continue
            mes_num, anio_num = int(m.group(1)), int(m.group(2))
            key = (anio_num, mes_num)
            if key not in meses:
                meses[key] = []
            meses[key].append(nombre)
    return meses


def cargar_precios_mes(zip_path: Path, archivos: list, eans_set: set) -> pd.DataFrame:
    """
    Lee los archivos de UN MES desde el ZIP y retorna precios de la canasta.

    Proceso:
    1. Extrae cada CSV.gz a disco (streaming, sin cargar todo en RAM)
    2. Lee en chunks de 200k filas
    3. Filtra INMEDIATAMENTE a eans_set (reduce a <0.1% de las filas)
    4. Derrite columnas precio_YYYYMMDD → (id_producto, anio_mes, precio)

    Returns: DataFrame[id_producto, anio_mes, precio] — precios CRUDOS (sin factor)
    """
    all_rows = []

    for archivo in sorted(archivos):
        tmp_path = TMP_DIR / Path(archivo).name

        # Extraer a disco en streaming
        with zipfile.ZipFile(zip_path) as z:
            with z.open(archivo) as src, open(tmp_path, 'wb') as dst:
                shutil.copyfileobj(src, dst, length=4 * 1024 * 1024)

        with gzip.open(tmp_path, 'rt', encoding='utf-8') as g:
            for chunk in pd.read_csv(
                g,
                dtype={'id_producto': 'str'},
                chunksize=200_000,
                low_memory=False
            ):
                # Normalizar a EAN-13
                chunk['id_producto'] = chunk['id_producto'].str.strip().str.zfill(13)

                # Filtrar a EANs de la canasta (reducción masiva de RAM)
                mask = chunk['id_producto'].isin(eans_set)
                if mask.sum() == 0:
                    continue
                sub = chunk.loc[mask].copy()

                price_cols = [c for c in sub.columns if c.startswith('precio_')]
                if not price_cols:
                    continue

                # Reemplazar 'NA' string y convertir a float
                sub_p = sub[['id_producto'] + price_cols].copy()
                for pc in price_cols:
                    sub_p[pc] = pd.to_numeric(
                        sub_p[pc].replace('NA', np.nan), errors='coerce'
                    )

                # Melt: wide → long
                melted = sub_p.melt(
                    id_vars='id_producto',
                    value_vars=price_cols,
                    var_name='col_fecha',
                    value_name='precio'
                )
                melted = melted[(melted['precio'].notna()) & (melted['precio'] > 0)]
                if len(melted) == 0:
                    continue

                # Extraer año-mes: 'precio_20260401' → '2026-04'
                fechas_str = melted['col_fecha'].str.replace('precio_', '', regex=False)
                melted = melted.copy()
                melted['anio_mes'] = (
                    fechas_str.str[:4] + '-' + fechas_str.str[4:6]
                )

                all_rows.append(melted[['id_producto', 'anio_mes', 'precio']])

        tmp_path.unlink(missing_ok=True)

    if not all_rows:
        return pd.DataFrame(columns=['id_producto', 'anio_mes', 'precio'])

    return pd.concat(all_rows, ignore_index=True)

## 4. Procesamiento de todos los semestres

Para cada semestre disponible en `SEPA_DIR`: carga mes a mes, filtra a los 51 EANs, detecta el factor de escala (centavos vs pesos) y agrega a mediana nacional mensual por producto.

El resultado `df_precios` tiene una fila por `(id_producto, anio_mes)` con la mediana nacional de precio.

In [ ]:
_CACHE_PRECIOS = CACHE_DIR / 'precios_canasta_mensual.parquet'

if USE_CACHE and _CACHE_PRECIOS.exists():
    print(f'Cargando caché: {_CACHE_PRECIOS}')
    df_precios = pd.read_parquet(_CACHE_PRECIOS)
    print(f'  {len(df_precios):,} filas cargadas')
    print(f'  Período: {df_precios["anio_mes"].min()} → {df_precios["anio_mes"].max()}')
    print(f'  EANs con datos: {df_precios["id_producto"].nunique()} / {len(EANS_SET)}')

else:
    semestres = detectar_semestres(SEPA_DIR)
    if not semestres:
        raise RuntimeError(f'No se encontraron ZIPs semestrales en {SEPA_DIR}')

    print(f'Semestres encontrados ({len(semestres)}):')
    for zp, lbl in semestres:
        print(f'  {lbl} → {zp.name}  ({zp.stat().st_size / 1024**3:.2f} GB)')

    registros = []

    for zip_path, semestre_label in semestres:
        print(f'\n{"─"*65}')
        print(f'Procesando semestre: {semestre_label}')

        meses = _archivos_por_mes(zip_path)
        meses_str = sorted(f'{a}-{m:02d}' for a, m in meses.keys())
        print(f'  Meses en ZIP: {meses_str}')

        for (anio, mes), archivos in tqdm(sorted(meses.items()),
                                           desc=f'  {semestre_label}', leave=False):
            anio_mes_label = f'{anio}-{mes:02d}'

            df_mes = cargar_precios_mes(zip_path, archivos, EANS_SET)

            if len(df_mes) == 0:
                print(f'  {anio_mes_label}: SIN DATOS para la canasta')
                continue

            # Autodetectar factor de escala: mediana > $10.000 → datos en centavos
            mediana_raw = df_mes['precio'].median()
            factor = 100 if mediana_raw > 10_000 else 1
            if factor == 100:
                df_mes = df_mes.copy()
                df_mes['precio'] = df_mes['precio'] / 100

            # Agregar: mediana nacional por (id_producto, anio_mes)
            # Nota: agrupa TODOS los reportes del país (sin distinción de cadena/provincia)
            df_agg = (
                df_mes.groupby(['id_producto', 'anio_mes'])
                .agg(
                    precio_mediano = ('precio', 'median'),
                    precio_p25     = ('precio', lambda x: x.quantile(0.25)),
                    precio_p75     = ('precio', lambda x: x.quantile(0.75)),
                    n_obs          = ('precio', 'count'),
                )
                .reset_index()
            )

            registros.append(df_agg)
            n_prod = df_mes['id_producto'].nunique()
            print(f'  {anio_mes_label}: {n_prod}/{len(EANS_SET)} EANs '
                  f'| factor={factor} | mediana ref=${df_mes["precio"].median():,.0f}')

            del df_mes; gc.collect()

    df_precios = pd.concat(registros, ignore_index=True) if registros else pd.DataFrame(
        columns=['id_producto', 'anio_mes', 'precio_mediano', 'precio_p25', 'precio_p75', 'n_obs']
    )

    if USE_CACHE and len(df_precios) > 0:
        df_precios.to_parquet(_CACHE_PRECIOS, compression='snappy', index=False)
        print(f'\nCaché guardado: {_CACHE_PRECIOS}')

    print(f'\n=== Resultado ===')
    print(f'  Filas totales: {len(df_precios):,}')
    print(f'  Período:       {df_precios["anio_mes"].min()} → {df_precios["anio_mes"].max()}')
    print(f'  EANs con datos: {df_precios["id_producto"].nunique()} / {len(EANS_SET)}')
    
    # Productos sin ningún dato
    eans_sin_datos = EANS_SET - set(df_precios['id_producto'].unique())
    if eans_sin_datos:
        print(f'  ⚠️  EANs SIN datos en ningún período ({len(eans_sin_datos)}):')
        for ean in sorted(eans_sin_datos):
            print(f'      {ean} — {EAN_DESC.get(ean, "desc. desconocida")}')

## 5. Verificación de cobertura

Heatmap de disponibilidad: qué productos tienen datos en qué meses. Permite identificar:
- Productos nuevos que no existían en 2024
- Productos que desaparecen del SEPA en algún período
- Gaps a tener en cuenta al interpretar el costo total de la canasta

In [ ]:
# Pivot de cobertura: id_producto × anio_mes → 1/0
pivot_cov = (
    df_precios
    .assign(tiene_dato=1)
    .pivot_table(
        index='id_producto',
        columns='anio_mes',
        values='tiene_dato',
        aggfunc='max',
        fill_value=0
    )
)

# Label legible: últimos 35 chars de descripción + 4 últimos dígitos del EAN
pivot_cov.index = pivot_cov.index.map(
    lambda ean: f"{EAN_DESC.get(ean, ean)[:38]} …({ean[-4:]})"
)

n_meses = pivot_cov.shape[1]
cobertura = pivot_cov.sum(axis=1).sort_values(ascending=False)
print(f'=== Cobertura por producto ({n_meses} meses disponibles) ===')
for label, cnt in cobertura.items():
    bar = '█' * cnt + '░' * (n_meses - cnt)
    print(f'  {cnt:2d}/{n_meses}  {bar}  {label}')

# Heatmap
fig, ax = plt.subplots(figsize=(max(12, n_meses * 0.55), max(14, len(pivot_cov) * 0.38)))
sns.heatmap(
    pivot_cov,
    cmap='RdYlGn',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Datos disponibles (1=sí)', 'shrink': 0.4},
    vmin=0, vmax=1,
    ax=ax
)
ax.set_title('Cobertura mensual de la canasta — verde=datos disponibles', fontsize=13, pad=12)
ax.set_xlabel('Período')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_cobertura_canasta.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {OUTPUT_DIR}/06_cobertura_canasta.png')

## 6. Cálculo del costo mensual de la canasta

Para cada `(EAN, mes)`: `precio_mediano × cantidad_mensual`.
Suma por mes = costo total de la canasta ese mes.

In [ ]:
# Enriquecer df_precios con metadata de la canasta
df_costo = df_precios.copy()
df_costo['cantidad']  = df_costo['id_producto'].map(EAN_CANTIDAD)
df_costo['categoria'] = df_costo['id_producto'].map(EAN_CAT)
df_costo['descripcion'] = df_costo['id_producto'].map(EAN_DESC)
df_costo['costo_item'] = df_costo['precio_mediano'] * df_costo['cantidad']

# Costo total de la canasta por mes
costo_total = (
    df_costo.groupby('anio_mes')
    .agg(
        costo_canasta    = ('costo_item',    'sum'),
        n_prod_con_datos = ('id_producto',   'nunique'),
    )
    .reset_index()
)
costo_total['anio_mes_dt'] = pd.to_datetime(costo_total['anio_mes'] + '-01')
costo_total = costo_total.sort_values('anio_mes_dt').reset_index(drop=True)

# Variaciones
costo_total['var_mensual_pct'] = costo_total['costo_canasta'].pct_change() * 100
costo_total['var_anual_pct']   = costo_total['costo_canasta'].pct_change(12) * 100
costo_total['indice']          = costo_total['costo_canasta'] / costo_total['costo_canasta'].iloc[0] * 100

print('=== Costo mensual de la canasta (51 productos, familia tipo 4 integrantes) ===')
print(costo_total[['anio_mes', 'costo_canasta', 'n_prod_con_datos',
                   'var_mensual_pct', 'var_anual_pct', 'indice']].to_string(index=False))

print(f'\nÚltimo mes disponible: {costo_total["anio_mes"].max()}')
print(f'Costo canasta: ${costo_total["costo_canasta"].iloc[-1]:,.0f}')

## 7. Visualizaciones

- **Panel superior:** costo nominal en pesos
- **Panel inferior:** variación mensual %
- **Stacked area:** composición por categoría
- **Índice base:** evolución normalizada a primer mes = 100

In [ ]:
# ── Gráfico 1: Costo nominal + variación mensual ──────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('Canasta Representativa — Familia Tipo 4 Integrantes (51 productos)',
             fontsize=14, fontweight='bold')

# Panel A: costo nominal
ax1 = axes[0]
x = costo_total['anio_mes_dt']
y = costo_total['costo_canasta']
ax1.plot(x, y, marker='o', linewidth=2.5, color='steelblue', markersize=6, zorder=3)
ax1.fill_between(x, y, alpha=0.12, color='steelblue')

# Anotaciones en los últimos 3 meses
for _, row in costo_total.tail(3).iterrows():
    ax1.annotate(
        f'${row["costo_canasta"]:,.0f}',
        xy=(row['anio_mes_dt'], row['costo_canasta']),
        xytext=(0, 13), textcoords='offset points',
        ha='center', fontsize=9, fontweight='bold', color='steelblue'
    )

ax1.set_title('Costo mensual nominal (ARS corrientes)', fontsize=12)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax1.grid(True, alpha=0.4)
ax1.set_xlabel('')

# Panel B: variación mensual %
ax2 = axes[1]
colores = ['crimson' if v >= 0 else 'seagreen'
           for v in costo_total['var_mensual_pct'].fillna(0)]
ax2.bar(x, costo_total['var_mensual_pct'], color=colores, alpha=0.75, width=20)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Variación mensual del costo (%)', fontsize=12)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:+.1f}%'))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax2.grid(True, alpha=0.4, axis='y')
ax2.set_xlabel('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_evolucion_costo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Gráfico 2: Composición por categoría (stacked area) ──────────────────────
costo_cat = (
    df_costo.groupby(['anio_mes', 'categoria'])
    .agg(costo_cat=('costo_item', 'sum'))
    .reset_index()
)
costo_cat['anio_mes_dt'] = pd.to_datetime(costo_cat['anio_mes'] + '-01')

# Orden de categorías por costo promedio descendente
orden_cat = (
    costo_cat.groupby('categoria')['costo_cat']
    .mean().sort_values(ascending=False).index.tolist()
)

pivot_cat = (
    costo_cat.pivot_table(
        index='anio_mes_dt', columns='categoria',
        values='costo_cat', aggfunc='sum'
    )[orden_cat]
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(15, 7))
paleta = plt.cm.tab20.colors[:len(orden_cat)]
ax.stackplot(pivot_cat.index, pivot_cat.values.T,
             labels=orden_cat, colors=paleta, alpha=0.82)
ax.set_title('Composición del costo por categoría (apilado)', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_canasta_por_categoria.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen del último mes
ultimo_mes = costo_cat['anio_mes'].max()
resumen_cat = (
    costo_cat[costo_cat['anio_mes'] == ultimo_mes]
    .sort_values('costo_cat', ascending=False)
    .assign(pct=lambda df: df['costo_cat'] / df['costo_cat'].sum() * 100)
)
print(f'\n=== Desglose por categoría — {ultimo_mes} ===')
print(resumen_cat[['categoria', 'costo_cat', 'pct']].to_string(index=False))
print(f'\nTotal canasta: ${resumen_cat["costo_cat"].sum():,.0f}')

In [ ]:
# ── Gráfico 3: Índice de precios (base = primer mes con datos = 100) ──────────
fig, ax = plt.subplots(figsize=(14, 6))

base_label = costo_total['anio_mes'].iloc[0]
ax.plot(
    costo_total['anio_mes_dt'],
    costo_total['indice'],
    marker='o', linewidth=2.5, color='darkorange', markersize=5
)
ax.axhline(100, color='gray', linewidth=0.8, linestyle=':', label=f'Base {base_label} = 100')

# Anotación del último valor
ult = costo_total.iloc[-1]
ax.annotate(
    f'{ult["indice"]:.0f}',
    xy=(ult['anio_mes_dt'], ult['indice']),
    xytext=(8, 0), textcoords='offset points',
    va='center', fontsize=11, fontweight='bold', color='darkorange'
)

ax.set_title(f'Índice de costo de la canasta (base {base_label} = 100)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '09_indice_canasta.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen de inflación acumulada
primero = costo_total.iloc[0]
ultimo  = costo_total.iloc[-1]
n_meses = len(costo_total) - 1
print(f'Período: {primero["anio_mes"]} → {ultimo["anio_mes"]} ({n_meses} meses)')
print(f'Costo inicial: ${primero["costo_canasta"]:,.0f}')
print(f'Costo final:   ${ultimo["costo_canasta"]:,.0f}')
print(f'Inflación acumulada canasta: {(ultimo["indice"] - 100):.1f}%')
if n_meses > 0:
    tma = ((ultimo['costo_canasta'] / primero['costo_canasta']) ** (12 / n_meses) - 1) * 100
    print(f'Tasa mensual anualizada:     {tma:.1f}% anual')

## 8. Comparación con IPC INDEC

Descarga el IPC Nacional (nivel general) desde la API oficial de datos del gobierno argentino y compara con el índice de la canasta, ambos normalizados al mismo mes base.

> **Nota:** el IPC INDEC está disponible con 1-2 meses de rezago. La API puede tardar unos segundos en responder.

In [ ]:
# ── IPC Nacional INDEC vía apis.datos.gob.ar ─────────────────────────────────
# Serie: 148.3_INIVELGENERAL_DICI_M_26 → IPC nivel general, base dic 2016 = 100
try:
    fecha_inicio = costo_total['anio_mes'].min()[:7]   # 'YYYY-MM'
    url_ipc = (
        'https://apis.datos.gob.ar/series/api/series/'
        f'?ids=148.3_INIVELGENERAL_DICI_M_26'
        f'&start_date={fecha_inicio}&format=csv'
    )
    print(f'Descargando IPC INDEC: {url_ipc}')
    df_ipc_raw = pd.read_csv(url_ipc, skiprows=3, names=['fecha', 'ipc'])
    df_ipc_raw = df_ipc_raw.dropna()
    df_ipc_raw['fecha'] = pd.to_datetime(df_ipc_raw['fecha'], errors='coerce')
    df_ipc_raw = df_ipc_raw.dropna(subset=['fecha'])
    df_ipc_raw['anio_mes'] = df_ipc_raw['fecha'].dt.strftime('%Y-%m')
    df_ipc_raw['ipc'] = pd.to_numeric(df_ipc_raw['ipc'], errors='coerce')
    df_ipc_raw = df_ipc_raw[['anio_mes', 'ipc']].dropna()
    print(f'IPC disponible: {df_ipc_raw["anio_mes"].min()} → {df_ipc_raw["anio_mes"].max()}')

    # Merge y normalización
    df_comp = costo_total[['anio_mes', 'costo_canasta', 'indice']].merge(
        df_ipc_raw, on='anio_mes', how='left'
    )
    # Normalizar IPC al mismo mes base que la canasta
    ipc_base = df_comp['ipc'].dropna().iloc[0]
    df_comp['idx_ipc'] = df_comp['ipc'] / ipc_base * 100
    df_comp['anio_mes_dt'] = pd.to_datetime(df_comp['anio_mes'] + '-01')

    primer_mes = df_comp['anio_mes'].iloc[0]
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df_comp['anio_mes_dt'], df_comp['indice'],
            marker='o', linewidth=2.5, label='Canasta SEPA (51 productos)', color='steelblue')
    ax.plot(df_comp['anio_mes_dt'], df_comp['idx_ipc'],
            marker='s', linewidth=2.5, label='IPC INDEC Nacional', color='darkorange',
            linestyle='--')
    ax.axhline(100, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)
    ax.set_title(f'Índice de precios: Canasta SEPA vs IPC INDEC (base {primer_mes} = 100)',
                 fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '10_canasta_vs_ipc.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n=== Índices comparados ===')
    print(df_comp[['anio_mes', 'indice', 'idx_ipc']].dropna(subset=['idx_ipc']).to_string(index=False))

except Exception as e:
    print(f'No se pudo obtener IPC INDEC: {e}')
    print('Continuar sin comparación. Verificar conexión a internet.')

## 9. Exportación a Excel

El archivo de salida `evolucion_canasta_representativa.xlsx` contiene tres hojas:

| Hoja | Contenido |
|------|-----------|
| **Evolución** | Costo total mensual, índice, variación mensual y anual |
| **Costo por producto** | Costo mensual de cada ítem (precio × cantidad) |
| **Precios** | Precio mediano mensual por producto (sin multiplicar por cantidad) |

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import re as _re

_HDR_FILL = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
_HDR_FONT = Font(bold=True, color='FFFFFF', size=10)
_HDR_ALIG = Alignment(horizontal='center', wrap_text=True, vertical='center')


def _fmt_header(ws, ancho_desc=42):
    ws.row_dimensions[1].height = 32
    ws.freeze_panes = 'A2'
    for cell in ws[1]:
        cell.fill = _HDR_FILL
        cell.font = _HDR_FONT
        cell.alignment = _HDR_ALIG


# ── Hoja Evolución ─────────────────────────────────────────────────────────────
evol_export = costo_total[[
    'anio_mes', 'costo_canasta', 'n_prod_con_datos',
    'var_mensual_pct', 'var_anual_pct', 'indice'
]].copy()

# ── Hoja Precios ───────────────────────────────────────────────────────────────
precio_pivot = (
    df_precios
    .pivot_table(
        index='id_producto',
        columns='anio_mes',
        values='precio_mediano',
        aggfunc='median'
    )
    .reset_index()
)
precio_pivot.insert(1, 'descripcion', precio_pivot['id_producto'].map(EAN_DESC))
precio_pivot.insert(2, 'categoria',   precio_pivot['id_producto'].map(EAN_CAT))
precio_pivot.insert(3, 'cantidad_mes', precio_pivot['id_producto'].map(EAN_CANTIDAD))

# ── Hoja Costo por producto ────────────────────────────────────────────────────
costo_pivot = precio_pivot.copy()
meses_cols = [c for c in costo_pivot.columns if _re.match(r'^\d{4}-\d{2}$', str(c))]
for mes in meses_cols:
    costo_pivot[mes] = costo_pivot[mes] * costo_pivot['cantidad_mes']

# ── Exportar ──────────────────────────────────────────────────────────────────
out_excel = OUTPUT_DIR / 'evolucion_canasta_representativa.xlsx'

with pd.ExcelWriter(out_excel, engine='openpyxl') as writer:
    evol_export.to_excel(writer,  sheet_name='Evolución',          index=False)
    costo_pivot.to_excel(writer,  sheet_name='Costo por producto', index=False)
    precio_pivot.to_excel(writer, sheet_name='Precios',            index=False)

    for sheet_name in ['Evolución', 'Costo por producto', 'Precios']:
        ws = writer.sheets[sheet_name]
        _fmt_header(ws)

        # Ancho de columnas base
        ws.column_dimensions['A'].width = 12
        for col_idx in range(2, ws.max_column + 1):
            col_letter = get_column_letter(col_idx)
            header_val = str(ws.cell(1, col_idx).value or '')
            if header_val in ('descripcion',):
                ws.column_dimensions[col_letter].width = 42
            elif header_val in ('categoria',):
                ws.column_dimensions[col_letter].width = 22
            else:
                ws.column_dimensions[col_letter].width = 13

        # Formato numérico para columnas de precio/costo
        for col_idx in range(1, ws.max_column + 1):
            col_val = str(ws.cell(1, col_idx).value or '')
            if _re.match(r'^\d{4}-\d{2}$', col_val) or 'costo' in col_val:
                for row in ws.iter_rows(min_row=2, max_row=ws.max_row,
                                        min_col=col_idx, max_col=col_idx):
                    for cell in row:
                        cell.number_format = '#,##0.00'
            elif 'pct' in col_val or 'var' in col_val:
                for row in ws.iter_rows(min_row=2, max_row=ws.max_row,
                                        min_col=col_idx, max_col=col_idx):
                    for cell in row:
                        cell.number_format = '+0.00"%"'

print(f'Excel exportado: {out_excel}')
print()
print('=' * 65)
print('RESUMEN FINAL — CANASTA REPRESENTATIVA')
print('=' * 65)
primero = costo_total.iloc[0]
ultimo  = costo_total.iloc[-1]
print(f'Período:           {primero["anio_mes"]} → {ultimo["anio_mes"]}')
print(f'Productos seguidos: {len(EANS_SET)}')
print(f'Costo {primero["anio_mes"]}:  ${primero["costo_canasta"]:>12,.0f}')
print(f'Costo {ultimo["anio_mes"]}:  ${ultimo["costo_canasta"]:>12,.0f}')
if primero['costo_canasta'] > 0:
    acum = (ultimo['costo_canasta'] / primero['costo_canasta'] - 1) * 100
    print(f'Inflación acumulada: {acum:>10.1f}%')
print('=' * 65)